In [12]:
import os
import sys
import pandas as pd
import re
from collections import Counter

# 환경 설정
project_dir = "/data/ephemeral/home/nlp-5/eunbyul/joe"
sys.path.append(project_dir)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### dev.csv 기준 비교 코드

In [13]:
import pandas as pd

DEV_CSV_PATH = "/data/ephemeral/home/nlp-5/eunbyul/joe/data/dev.csv"
DEV_SUB_PATH = "/data/ephemeral/home/nlp-5/eunbyul/joe/prediction/dev_submission_digit82-kobart-summarization_beam4_len165_20250804_0257.csv"

df_gt = pd.read_csv(DEV_CSV_PATH)
df_pred = pd.read_csv(DEV_SUB_PATH)

# merge: 같은 컬럼명(summary) -> summary_x(정답), summary_y(예측)
df_merged = pd.merge(df_gt, df_pred, on="fname", how="inner")

# 컬럼명 확인
print("컬럼명:", df_merged.columns.tolist())
# ['fname', 'dialogue', 'summary_x', 'topic', 'summary_y']

# 비교 함수
def is_same(a, b):
    return str(a).strip() == str(b).strip()

df_merged['is_exact_match'] = df_merged.apply(lambda row: is_same(row['summary_x'], row['summary_y']), axis=1)
df_merged['gt_len'] = df_merged['summary_x'].apply(len)
df_merged['pred_len'] = df_merged['summary_y'].apply(len)

# 틀린 샘플 100개만 저장
wrong_cases = df_merged[~df_merged['is_exact_match']].copy()
sample_wrong = wrong_cases[['fname', 'dialogue', 'summary_x', 'summary_y', 'gt_len', 'pred_len']].head(100)
sample_wrong = sample_wrong.rename(columns={'summary_x': 'gt_summary', 'summary_y': 'pred_summary'})

sample_wrong.to_csv("dev_sample_wrong_case_20250804_0257.csv", index=False, encoding='utf-8-sig')
print("일치하지 않는 dev 샘플 100개 저장 완료!")

print(f"전체 {len(df_merged)}개 중 정답 일치 비율: {df_merged['is_exact_match'].mean():.2%}")
print(f"평균 정답 요약 길이: {df_merged['gt_len'].mean():.1f}")
print(f"평균 모델 요약 길이: {df_merged['pred_len'].mean():.1f}")

for i, row in sample_wrong.iterrows():
    print(f"---- fname: {row['fname']} ----")
    print("▶ DIALOGUE:", row['dialogue'][:150], "...")
    print("▶ 정답:", row['gt_summary'])
    print("▶ 예측:", row['pred_summary'])
    print(f"▶ 길이(정답/예측): {row['gt_len']}/{row['pred_len']}")
    print("="*60)


컬럼명: ['fname', 'dialogue', 'summary_x', 'topic', 'summary_y']
일치하지 않는 dev 샘플 100개 저장 완료!
전체 499개 중 정답 일치 비율: 0.00%
평균 정답 요약 길이: 81.2
평균 모델 요약 길이: 80.3
---- fname: dev_0 ----
▶ DIALOGUE: #Person1#: 안녕하세요, 오늘 기분이 어떠세요?
#Person2#: 요즘 숨쉬기가 힘들어요.
#Person1#: 최근에 감기에 걸렸나요?
#Person2#: 아니요, 감기는 안 걸렸어요. 숨쉴 때 가슴이 답답해요.
#Person1#: 혹시 알고 있는 알레르기 있 ...
▶ 정답: #Person2#는 숨쉬기 어려워합니다. 의사는 #Person2#에게 증상을 확인하고, 천식 검사를 위해 폐 전문의에게 가볼 것을 권합니다.
▶ 예측: #Person1# 은 숨쉬기가 힘들고 천식을 앓고 있습니다. #Person2# 는 천식 검사를 위해 폐 전문의에게 가보라고 권유합니다.
▶ 길이(정답/예측): 78/75
---- fname: dev_1 ----
▶ DIALOGUE: #Person1#: 야 Jimmy, 오늘 좀 이따 운동하러 가자.
#Person2#: 그래, 몇 시에 갈래?
#Person1#: 3시 30분 어때?
#Person2#: 좋아. 오늘은 다리랑 팔 운동하는 날이야.
#Person1#: 근데 나 아까 농구해서 다리가 좀 아파 ...
▶ 정답: #Person1#는 Jimmy를 운동하러 초대하고 팔과 복근 운동을 하도록 설득합니다.
▶ 예측: #Person1# 은 Jimmy에게 운동 날짜를 알려주고, #Person2# 는 #Person1# 에게 오후 3시 30분에 체육관에서 운동하자고 제안한다.
▶ 길이(정답/예측): 48/86
---- fname: dev_2 ----
▶ DIALOGUE: #Person1#: 나 진짜 건강에 안 좋은 음식 좀 그만 먹어야겠어. 
#Person2#: 맞아, 무슨 말인지 알아. 나도 요즘 건강하게 먹으려

In [ ]:
import pandas as pd

TRAIN_CSV_PATH = "/data/ephemeral/home/nlp-5/eunbyul/joe/data/train.csv"
TRAIN_SUB_PATH = "/data/ephemeral/home/nlp-5/eunbyul/joe/prediction/dev_submission_digit82-kobart-summarization_beam4_len165_20250804_0344.csv"

df_gt = pd.read_csv(TRAIN_CSV_PATH)
df_pred = pd.read_csv(TRAIN_SUB_PATH)

# merge: 같은 컬럼명(summary) -> summary_x(정답), summary_y(예측)
df_merged = pd.merge(df_gt, df_pred, on="fname", how="inner")

# 컬럼명 확인
print("컬럼명:", df_merged.columns.tolist())
# ['fname', 'dialogue', 'summary_x', 'topic', 'summary_y']

# 비교 함수
def is_same(a, b):
    return str(a).strip() == str(b).strip()

df_merged['is_exact_match'] = df_merged.apply(lambda row: is_same(row['summary_x'], row['summary_y']), axis=1)
df_merged['gt_len'] = df_merged['summary_x'].apply(len)
df_merged['pred_len'] = df_merged['summary_y'].apply(len)

# 틀린 샘플 100개만 저장
wrong_cases = df_merged[~df_merged['is_exact_match']].copy()
sample_wrong = wrong_cases[['fname', 'dialogue', 'summary_x', 'summary_y', 'gt_len', 'pred_len']].head(100)
sample_wrong = sample_wrong.rename(columns={'summary_x': 'gt_summary', 'summary_y': 'pred_summary'})

sample_wrong.to_csv("sample_wrong_case_20250804_0344.csv", index=False, encoding='utf-8-sig')
print("일치하지 않는 dev 샘플 100개 저장 완료!")

print(f"전체 {len(df_merged)}개 중 정답 일치 비율: {df_merged['is_exact_match'].mean():.2%}")
print(f"평균 정답 요약 길이: {df_merged['gt_len'].mean():.1f}")
print(f"평균 모델 요약 길이: {df_merged['pred_len'].mean():.1f}")

for i, row in sample_wrong.iterrows():
    print(f"---- fname: {row['fname']} ----")
    print("▶ DIALOGUE:", row['dialogue'][:150], "...")
    print("▶ 정답:", row['gt_summary'])
    print("▶ 예측:", row['pred_summary'])
    print(f"▶ 길이(정답/예측): {row['gt_len']}/{row['pred_len']}")
    print("="*60)

### 차이점 비교

In [6]:
import difflib

def diff_text(a, b):
    # 두 문장의 유사도 및 라인 단위 차이 출력
    a_lines = a.split()
    b_lines = b.split()
    diff = difflib.unified_diff(a_lines, b_lines, lineterm='', n=0)
    return '\n'.join(diff)

for i, row in sample_df.iterrows():
    print(f"\n==== 샘플 {i+1} ====")
    print("[정답 요약]\n", row['summary'])
    print("\n[내 요약]\n", row[sub_sum_col])
    print('\n[Diff]')
    print(diff_text(str(row['summary']), str(row[sub_sum_col])))
    print('-'*90)


==== 샘플 461 ====
[정답 요약]
 #Person1#은 #Person2#에게 어떤 남자와 거리를 두라고 권유하고, #Person2#도 동의한다.

[내 요약]
 #Person1#은 #Person2#에게 어떤 남자와 거리를 두라고 권유하고, #Person2#도 동의한다.

[Diff]

------------------------------------------------------------------------------------------

==== 샘플 74 ====
[정답 요약]
 #Person1#은 다운타임으로 인한 생산 손실을 줄이기 위해 유지보수 절차를 확립하자고 제안합니다.

[내 요약]
 #Person1#은 다운타임으로 인한 생산 손실을 줄이기 위해 유지보수 절차를 확립하자고 제안합니다.

[Diff]

------------------------------------------------------------------------------------------

==== 샘플 232 ====
[정답 요약]
 Bobby는 Dr. Cardano에게 오른쪽 발에 날카로운 통증이 생겼다고 설명합니다. Dr. Cardano는 Bobby의 발 상태를 확인한 후 실험실에서 혈액 검사를 받도록 지시합니다.

[내 요약]
 Bobby는 Dr. Cardano에게 오른쪽 발에 날카로운 통증이 생겼다고 설명합니다. Dr. Cardano는 Bobby의 발 상태를 확인한 후 실험실에서 혈액 검사를 받도록 지시합니다.

[Diff]

------------------------------------------------------------------------------------------

==== 샘플 176 ====
[정답 요약]
 #Person1#과 #Person2#은 매니저가 직원의 업무 불만족에 대해 개인적으로 대화하며 긍정적인 해결책을 찾는 것이 중요하다고 논의한다.

[내 요약]
 #Person1#과 #Person2#은 매니저가 직원의

### 인물, 지시어, 토큰, 숫자, 주요 단어만 비교